# Baseline Functionalities

In this notebook, we will demonstrate the baseline functionalities of the LatteReview package. These functionalities are the cornerstone of reviewer agents.

## Setting up the notebook

In [1]:
%reload_ext autoreload
%autoreload 2

from dotenv import load_dotenv

# Load environment variables from .env file. Adjust the path to the .env file as needed.
load_dotenv(dotenv_path='../../.env')

# Enable asyncio in Jupyter
import asyncio
import nest_asyncio

nest_asyncio.apply()

#  Add the package to the path (required if you are running this notebook from the examples folder)
import sys
sys.path.append('../../')


High-level configs

In [2]:
%reload_ext autoreload
%autoreload 2

from dotenv import load_dotenv

# Load environment variables from .env file. Adjust the path to the .env file as needed.
load_dotenv(dotenv_path='../../.env')

# Enable asyncio in Jupyter
import asyncio
import nest_asyncio

nest_asyncio.apply()

#  Add the package to the path (required if you are running this notebook from the examples folder)
import sys
sys.path.append('../../')


Import required packages

In [3]:
import pandas as pd

from lattereview.providers import OpenAIProvider
from lattereview.providers import OllamaProvider
from lattereview.providers import LiteLLMProvider
from lattereview.agents import ScoringReviewer

## Testing the base functionalities

Testing the OpenAI provider (with current and older OpenAI models, and a Gemini model):

In [4]:
question = "What is the capital of France?"

# A current OpenAI model
openai_provider = OpenAIProvider(model="gpt-6-luna")
print(asyncio.run(openai_provider.get_response(question)))

# Older models keep working unchanged
legacy_provider = OpenAIProvider(model="gpt-4o-mini")
print(asyncio.run(legacy_provider.get_response(question)))

# Gemini models through Google's OpenAI-compatible endpoint (uses GEMINI_API_KEY)
gemini_provider = OpenAIProvider(model="gemini-3.8-flash")
print(asyncio.run(gemini_provider.get_response(question)))

('Paris.', {'input_cost': 2.3e-06, 'output_cost': 8e-06, 'total_cost': 1.03e-05})
('The capital of France is Paris.', {'input_cost': 3.6e-06, 'output_cost': 4.2e-06, 'total_cost': 7.8e-06})
('The capital of France is **Paris**.', {'input_cost': 1.125e-05, 'output_cost': 0.0002475, 'total_cost': 0.00025875})


Testing the Ollama provider:

In [5]:
ollama_provider = OllamaProvider(model="qwen3.8:27b", host="http://localhost:11434")
question = "What is the capital of France?"
asyncio.run(ollama_provider.get_response(question))

('The capital of France is **Paris**.',
 {'input_cost': 0, 'output_cost': 0, 'total_cost': 0})

Testing the LiteLLM provider:

In [6]:
litellm_provider = LiteLLMProvider(model="anthropic/claude-sonnet-5")

# Other models to try:
# litellm_provider = LiteLLMProvider(model="gemini/gemini-3.8-flash")
# litellm_provider = LiteLLMProvider(model="gpt-6-luna")
# litellm_provider = LiteLLMProvider(model="gpt-4o-mini")
# litellm_provider = LiteLLMProvider(model="groq/openai/gpt-oss-120b")
# litellm_provider = LiteLLMProvider(model="deepseek/deepseek-flash")
# litellm_provider = LiteLLMProvider(model="ollama_chat/qwen3.8:27b")

question = "What is the capital of France?"
asyncio.run(litellm_provider.get_response(question))

("The capital of France is **Paris**.\n\nParis has been the country's capital for centuries and is known for landmarks such as the Eiffel Tower, the Louvre Museum, Notre-Dame Cathedral, and the Champs-Élysées. It's also the political, economic, and cultural center of France.",
 0.00104)

Testing the ScoringReviewer agent:

In [7]:
# Load a dummy dataset

data = pd.read_csv("data.csv")
data

,Title,Abstract,Authors,Year
0,Fusing an agent-based model of mosquito popula...,The mosquito Aedes aegypti is the vector of a ...,"Cavany, S.M.",2022
1,PDRL: Multi-Agent based Reinforcement Learning...,Reinforcement learning has been increasingly a...,"Shaik, T.",2023
2,Learning-accelerated Discovery of Immune-Tumou...,We present an integrated framework for enablin...,"Ozik, J.",2019
3,Investigating spatiotemporal dynamics and sync...,"In this paper we present AceMod, an agent-base...","Cliff, O.M.",2018
4,Modeling the Spread of COVID-19 in University ...,Mathematical and simulation models are often u...,"Herrmann, J.W.",2024
5,Multi-Agent Reinforcement Learning with Action...,Unmanned Aerial Vehicles (UAVs) are increasing...,"Rizvi, D.",2023
6,A new (N) stochastic quenched disorder model f...,Human beings live in a networked world in whic...,"Ferreira, A.A.",2019
7,Frustration induced phases in migrating cell c...,Collective motion of cells is common in many p...,"Copenhagen, K.",2017
8,Universal masking is urgent in the COVID-19 pa...,We present two models for the COVID-19 pandemi...,"Kai, D.",2020
9,Calculus of consent via MARL: Legitimating the...,"Public policies that supply public goods, espe...","Hu, Y.",2021


In [8]:
agent = ScoringReviewer(
    provider=OllamaProvider(model="qwen3.8:27b", host="http://localhost:11434"),
    name="Pouria",
    max_concurrent_requests=1, 
    backstory="an expert reviewer and researcher!",
    input_description = "article title",
    model_args={"reasoning_effort": "low"},
    reasoning = "brief",
    scoring_task="Search for articles whose titles suggest they are about AI or machine learning agents.",
    scoring_set=[1, 2],
    scoring_rules='Score 1 if the article title meets the criteria, and 2 if the article paper does not meet the criteria.',
)

# Dummy review
import json

text_list = data.Title.str.lower().tolist()
results, total_cost = asyncio.run(agent.review_items(text_list))
print("\n====== Review Results ======")
for n, result in enumerate(results):
    result_dict = json.loads(result)
    print("Input: ", text_list[n])
    print("reasoning: ", result_dict["reasoning"])
    print("score: ", result_dict["score"])
    print("--------------------------------")

# Dummy costs
print("\n====== Total Cost ======")
print(total_cost)

Reviewing 20 items - 2026-09-27 15:33:58: 100%|██████████| 20/20 [20:16<00:00, 60.81s/it]


====== Review Results ======
Input:  fusing an agent-based model of mosquito population dynamics with a statistical reconstruction of spatio-temporal abundance patterns
reasoning:  The title refers to an 'agent-based model' in the context of mosquito population dynamics, which is a well-established simulation methodology in ecology and epidemiology where individual rule-based agents interact to produce population-level patterns; this is fundamentally different from AI or machine learning agents, so the title does not suggest the article is about AI/ML agents.
score:  2
--------------------------------
Input:  pdrl: multi-agent based reinforcement learning for predictive monitoring
reasoning:  The title explicitly mentions 'multi-agent' and 'reinforcement learning,' both of which are core concepts in AI and machine learning agents, clearly meeting the criteria.
score:  1
--------------------------------
Input:  learning-accelerated discovery of immune-tumour interactions
reasoning:  Th